In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
startup = pd.read_csv("startup_funding.csv")

In [3]:
print("Number of rows and columns:", startup.shape)
startup.head()

Number of rows and columns: (3044, 10)


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09-01-2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13-01-2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09-01-2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02-01-2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02-01-2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [4]:
startup.info()

<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Sr No              3044 non-null   int64
 1   Date dd/mm/yyyy    3044 non-null   str  
 2   Startup Name       3044 non-null   str  
 3   Industry Vertical  2873 non-null   str  
 4   SubVertical        2108 non-null   str  
 5   City  Location     2864 non-null   str  
 6   Investors Name     3020 non-null   str  
 7   InvestmentnType    3040 non-null   str  
 8   Amount in USD      2084 non-null   str  
 9   Remarks            419 non-null    str  
dtypes: int64(1), str(9)
memory usage: 593.3 KB


In [5]:
startup.isnull().sum()

Sr No                   0
Date dd/mm/yyyy         0
Startup Name            0
Industry Vertical     171
SubVertical           936
City  Location        180
Investors Name         24
InvestmentnType         4
Amount in USD         960
Remarks              2625
dtype: int64

In [6]:
startup.columns = startup.columns.str.strip()

startup = startup.rename(columns={
    "Sr No": "sr_no",
    "Date dd/mm/yyyy": "date_raw",
    "Startup Name": "startup_raw",
    "Industry Vertical": "industry_raw",
    "SubVertical": "sub_vertical",
    "City  Location": "city_raw",
    "Investors Name": "investors_raw",
    "InvestmentnType": "investment_type_raw",
    "Amount in USD": "amount_raw",
    "Remarks": "remarks",
})

startup.columns

Index(['sr_no', 'date_raw', 'startup_raw', 'industry_raw', 'sub_vertical',
       'city_raw', 'investors_raw', 'investment_type_raw', 'amount_raw',
       'remarks'],
      dtype='str')

### Clean text columns

In [7]:
text_columns = [
    "date_raw", "startup_raw", "industry_raw", "sub_vertical",
    "city_raw", "investors_raw", "investment_type_raw",
    "amount_raw", "remarks"
]

for column in text_columns:
    startup[column] = startup[column].astype("string")

    startup[column] = startup[column].str.replace(r"\\+xe2\\+x80\\+x99", "'", regex=True)
    startup[column] = startup[column].str.replace(r"\\+xe2\\+x80\\+x98", "'", regex=True)
    startup[column] = startup[column].str.replace(r"\\+xe2\\+x80\\+x9[cd]", '"', regex=True)
    startup[column] = startup[column].str.replace(r"\\+xe2\\+x80\\+x9[34]", "-", regex=True)
    startup[column] = startup[column].str.replace(r"\\+xc3\\+x98", "O", regex=True)
    startup[column] = startup[column].str.replace(r"\\+xc2\\+xa0", " ", regex=True)
    startup[column] = startup[column].str.replace(r"\\+n", " ", regex=True)
    startup[column] = startup[column].str.replace(r"\\+'", "'", regex=True)
    startup[column] = startup[column].str.replace(r"\\+x[0-9a-fA-F]{2}", "", regex=True)

    startup[column] = startup[column].str.replace("\\u00a0", " ", regex=False)
    startup[column] = startup[column].str.replace(r"\\s+", " ", regex=True).str.strip()
    startup[column] = startup[column].str.strip('"').str.strip()

    startup[column] = startup[column].replace(
        ["", "nan", "N/A", "n/a", "na", "NA", "-"], np.nan
    )

startup[["startup_raw", "industry_raw", "city_raw"]].head()

,startup_raw,industry_raw,city_raw
0,BYJU’S,E-Tech,Bengaluru
1,Shuttl,Transportation,Gurgaon
2,Mamaearth,E-commerce,Bengaluru
3,https://www.wealthbucket.in/,FinTech,New Delhi
4,Fashor,Fashion and Apparel,Mumbai


### Clean funding dates

In [8]:
startup["date_fixed"] = startup["date_raw"].str.replace(".", "/", regex=False)
startup["date_fixed"] = startup["date_fixed"].str.replace("-", "/", regex=False)
startup["date_fixed"] = startup["date_fixed"].str.replace(".", "/", regex=False)

# Fix dates written like 05/072018
date_mask = startup["date_fixed"].str.fullmatch(r"\d{1,2}/\d{2}\d{4}", na=False)
startup.loc[date_mask, "date_fixed"] = (
    startup.loc[date_mask, "date_fixed"]
    .str.replace(r"^(\d{1,2})/(\d{2})(\d{4})$", r"\1/\2/\3", regex=True)
)

# Fix years written with only three digits, such as 01/07/015
year_mask = startup["date_fixed"].str.fullmatch(r"\d{1,2}/\d{1,2}/\d{3}", na=False)
startup.loc[year_mask, "date_fixed"] = (
    startup.loc[year_mask, "date_fixed"]
    .str.replace(r"^(\d{1,2}/\d{1,2}/)(\d{3})$", r"\g<1>2\2", regex=True)
)

startup["funding_date"] = pd.to_datetime(
    startup["date_fixed"], format="%d/%m/%Y", errors="coerce"
)

startup[["date_raw", "date_fixed", "funding_date"]].head()

,date_raw,date_fixed,funding_date
0,09-01-2020,09/01/2020,2020-01-09
1,13-01-2020,13/01/2020,2020-01-13
2,09-01-2020,09/01/2020,2020-01-09
3,02-01-2020,02/01/2020,2020-01-02
4,02-01-2020,02/01/2020,2020-01-02


In [9]:
print("Rows with invalid or missing dates:", startup["funding_date"].isna().sum())

Rows with invalid or missing dates: 1


### Clean funding amounts

In [10]:
startup["amount_clean"] = (
    startup["amount_raw"]
    .str.replace(",", "", regex=False)
    .str.replace(r"[^0-9.]", "", regex=True)
)

startup["amount_usd"] = pd.to_numeric(startup["amount_clean"], errors="coerce")
startup.loc[startup["amount_usd"] <= 0, "amount_usd"] = np.nan

# The source notebook included this specific Rapido correction.
rapido_rows = (
    startup["startup_raw"].str.contains("Rapido", case=False, na=False)
    & (startup["amount_usd"] >= 3_000_000_000)
)
startup.loc[rapido_rows, "amount_usd"] = 52_000_000

startup["amount_disclosed"] = startup["amount_usd"].notna().astype(int)
startup["amount_usd_mn"] = (startup["amount_usd"] / 1_000_000).round(4)

startup[["startup_raw", "amount_raw", "amount_usd", "amount_usd_mn"]].head()

,startup_raw,amount_raw,amount_usd,amount_usd_mn
0,BYJU’S,"20,00,00,000",200000000.0,200.0
1,Shuttl,"80,48,394",8048394.0,8.0484
2,Mamaearth,"1,83,58,860",18358860.0,18.3589
3,https://www.wealthbucket.in/,"30,00,000",3000000.0,3.0
4,Fashor,"18,00,000",1800000.0,1.8


### Standardize startup names

In [11]:
startup["startup_tmp"] = startup["startup_raw"].str.replace(
    r"^https?://", "", case=False, regex=True
)
startup["startup_tmp"] = startup["startup_tmp"].str.replace(
    r"^www\\.", "", case=False, regex=True
)
startup["startup_tmp"] = startup["startup_tmp"].str.rstrip("/")
startup["startup_tmp"] = startup["startup_tmp"].str.replace(
    r"\\.(com|in|co|io|org|net)(\\.in)?$", "", case=False, regex=True
).str.strip()

startup["startup_key"] = startup["startup_tmp"].str.lower().str.replace(
    r"[^a-z0-9]", "", regex=True
)

startup[["startup_raw", "startup_tmp", "startup_key"]].head()

,startup_raw,startup_tmp,startup_key
0,BYJU’S,BYJU’S,byjus
1,Shuttl,Shuttl,shuttl
2,Mamaearth,Mamaearth,mamaearth
3,https://www.wealthbucket.in/,www.wealthbucket.in,wwwwealthbucketin
4,Fashor,Fashor,fashor


In [12]:
startup_name_map = (
    startup.groupby("startup_key")["startup_tmp"]
    .agg(lambda names: names.value_counts().index[0] if names.notna().any() else np.nan)
)

startup["startup_name"] = startup["startup_key"].map(startup_name_map)
lowercase_names = startup["startup_name"].notna() & startup["startup_name"].str.islower()
startup.loc[lowercase_names, "startup_name"] = startup.loc[lowercase_names, "startup_name"].str.title()

startup.loc[startup["startup_key"] == "byjus", "startup_name"] = "BYJU'S"
startup.loc[startup["startup_key"].isin(["oyorooms", "oyo"]), "startup_name"] = "OYO Rooms"
startup.loc[startup["startup_key"].isin(["ola", "olacabs"]), "startup_name"] = "Ola Cabs"

startup[["startup_raw", "startup_name"]].head(10)

,startup_raw,startup_name
0,BYJU’S,BYJU'S
1,Shuttl,Shuttl
2,Mamaearth,MamaEarth
3,https://www.wealthbucket.in/,Www.Wealthbucket.In
4,Fashor,Fashor
5,Pando,Pando
6,Zomato,Zomato
7,Ecozen,Ecozen
8,CarDekho,CarDekho
9,Dhruva Space,Dhruva Space


### Group industries into sectors

In [13]:
startup["industry_vertical"] = startup["industry_raw"]
startup["industry_lower"] = startup["industry_raw"].fillna("").str.lower()

startup["sector"] = "Others"
startup.loc[startup["industry_raw"].isna(), "sector"] = "Unknown"

sector_rules = [
    ("Consumer Internet", r"^consumer internet$"),
    ("FinTech & Financial Services", r"fin|payment|lend|loan|insur|wallet|bank|invest|credit|wealth|money|nbfc|crypto"),
    ("EdTech", r"edu|ed-?tech|e-?tech|learn|tutor|school|course|exam|skill"),
    ("HealthTech & Wellness", r"health|medic|doctor|pharma|hospital|clinic|fitness|wellness|diagnos|care"),
    ("Food & Beverage", r"food|restaurant|beverage|cafe|dining|kitchen|meal|tea|coffee|grocer"),
    ("E-commerce & Retail", r"e-?commerce|marketplace|retail|shop|store|fashion|apparel|jewel|furniture|consumer goods|fmcg|d2c"),
    ("Logistics & Mobility", r"logistic|transport|cab|taxi|mobility|bike|auto|vehicle|car |cars|ev\\b|electric|delivery|fleet|freight|shuttle"),
    ("Travel & Hospitality", r"travel|hotel|hospitality|tour|stay|trip|accommodation"),
    ("Media & Entertainment", r"media|content|entertain|video|music|news|gaming|game|social|sport|film|publish"),
    ("Real Estate & Home Services", r"real estate|property|housing|home|rent|construction|interior"),
    ("Technology, SaaS & IT", r"tech|saas|software|^it$|information technology|analytic|\\bai\\b|artificial|cloud|data|iot|internet of things|platform|app|b2b|enterprise|cyber|digital")
]

for sector_name, keyword_pattern in sector_rules:
    sector_matches = startup["industry_lower"].str.contains(keyword_pattern, regex=True, na=False)
    rows_not_classified_yet = startup["sector"].isin(["Others"])
    startup.loc[sector_matches & rows_not_classified_yet, "sector"] = sector_name

startup[["industry_raw", "sector"]].head(10)

,industry_raw,sector
0,E-Tech,EdTech
1,Transportation,Logistics & Mobility
2,E-commerce,E-commerce & Retail
3,FinTech,FinTech & Financial Services
4,Fashion and Apparel,E-commerce & Retail
5,Logistics,Logistics & Mobility
6,Hospitality,HealthTech & Wellness
7,Technology,"Technology, SaaS & IT"
8,E-Commerce,E-commerce & Retail
9,Aerospace,Others


### Standardize city names

In [14]:
startup["city_first"] = startup["city_raw"].str.split(r"/|,|&| and ", regex=True).str[0].str.strip()
startup["city_lower"] = startup["city_first"].fillna("").str.lower().str.strip()

city_map = {
    "bangalore": "Bengaluru", "bengaluru": "Bengaluru", "kormangala": "Bengaluru", "karnataka": "Bengaluru",
    "gurgaon": "Gurugram", "gurugram": "Gurugram", "haryana": "Gurugram",
    "new delhi": "New Delhi", "delhi": "New Delhi", "nw delhi": "New Delhi",
    "mumbai": "Mumbai", "andheri": "Mumbai", "chembur": "Mumbai",
    "chennai": "Chennai", "taramani": "Chennai",
    "ahmedabad": "Ahmedabad", "ahemadabad": "Ahmedabad", "ahemdabad": "Ahmedabad",
    "kolkata": "Kolkata", "kolkatta": "Kolkata",
    "bhubaneswar": "Bhubaneswar", "bhubneswar": "Bhubaneswar",
    "trivandrum": "Thiruvananthapuram", "panaji": "Goa", "missourie": "Mussoorie"
}

international_cities = [
    "san francisco", "san jose", "burnsville", "menlo park", "palo alto", "santa monica",
    "singapore", "nairobi", "new york", "california", "us", "usa", "london", "boston",
    "dallas", "seattle", "sfo", "tulangan"
]

startup["city"] = startup["city_lower"].map(city_map)
startup["city"] = startup["city"].fillna(startup["city_first"].str.title())
startup.loc[startup["city_lower"].isin(["india", ""]), "city"] = "Unknown"
startup.loc[startup["city_first"].isna(), "city"] = "Unknown"

startup.loc[startup["city_lower"] == "us", "city"] = "USA"
startup.loc[startup["city_lower"] == "usa", "city"] = "USA"
startup.loc[startup["city_lower"] == "sfo", "city"] = "San Francisco"

startup["is_international_hq"] = startup["city_lower"].isin(international_cities).astype(int)
startup["is_multi_city"] = startup["city_raw"].str.contains(r"/|,|&| and ", regex=True, na=False).astype(int)

startup[["city_raw", "city", "is_international_hq", "is_multi_city"]].head(10)

,city_raw,city,is_international_hq,is_multi_city
0,Bengaluru,Bengaluru,0,0
1,Gurgaon,Gurugram,0,0
2,Bengaluru,Bengaluru,0,0
3,New Delhi,New Delhi,0,0
4,Mumbai,Mumbai,0,0
5,Chennai,Chennai,0,0
6,Gurgaon,Gurugram,0,0
7,Pune,Pune,0,0
8,Gurgaon,Gurugram,0,0
9,Bengaluru,Bengaluru,0,0


### Standardize investment types into funding stages

In [15]:
startup["investment_type"] = startup["investment_type_raw"]
startup["investment_type_lower"] = (
    startup["investment_type_raw"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-z ]", " ", regex=True)
    .str.replace(r"\\s+", " ", regex=True)
    .str.strip()
)

startup["funding_stage"] = "Other (Venture / Corporate / Bridge)"
startup.loc[startup["investment_type_raw"].isna(), "funding_stage"] = "Unknown"

startup.loc[startup["investment_type_lower"].str.contains("debt", na=False), "funding_stage"] = "Debt"

pre_series = startup["investment_type_lower"].str.contains("pre series", na=False)
pre_seed = startup["investment_type_lower"].str.contains("pre seed", na=False)
startup.loc[pre_series, "funding_stage"] = "Pre-Series A"
startup.loc[pre_seed, "funding_stage"] = "Seed / Angel"

seed_or_angel = startup["investment_type_lower"].str.contains(r"seed|angel|angle", regex=True, na=False)
startup.loc[seed_or_angel & ~pre_series, "funding_stage"] = "Seed / Angel"

series_letter = startup["investment_type_lower"].str.extract(r"series ([a-z])", expand=False).str.upper()
startup.loc[series_letter.isin(["A", "B", "C"]), "funding_stage"] = "Series " + series_letter.fillna("")
startup.loc[series_letter.isin(["D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z"]), "funding_stage"] = "Series D+"

private_equity = startup["investment_type_lower"].str.contains("private equity", na=False)
equity_only = startup["investment_type_lower"].eq("equity")
startup.loc[private_equity | equity_only, "funding_stage"] = "Private Equity"

startup[["investment_type_raw", "funding_stage"]].head(10)

,investment_type_raw,funding_stage
0,Private Equity Round,Private Equity
1,Series C,Series C
2,Series B,Series B
3,Pre-series A,Series A
4,Seed Round,Seed / Angel
5,Series A,Series A
6,Private Equity Round,Private Equity
7,Series A,Series A
8,Series D,Series D+
9,Seed,Seed / Angel


### Clean investor names and count investors

In [16]:
startup["investors_name"] = startup["investors_raw"].fillna("Unknown")
investors_clean = startup["investors_name"].str.replace(r"\\(.*?\\)", "", regex=True)
investors_clean = investors_clean.str.replace(". ", ", ", regex=False)
investors_clean = investors_clean.str.replace(r"Michael\\s*(&|and)\\s*Susan Dell", "Michael-Susan Dell", case=False, regex=True)

# Standardize common investor name variations
investor_aliases = [
    (r"^sequoia( capital)?( india)?( advisors)?$", "Sequoia Capital"),
    (r"^accel( partners)?( india)?$", "Accel Partners"),
    (r"^tiger global( management)?$", "Tiger Global"),
    (r"^saif partners$", "SAIF Partners"),
    (r"^kalaari capital( partners)?$", "Kalaari Capital"),
    (r"^idg ventures( india)?$", "IDG Ventures"),
    (r"^matrix partners?( india)?$", "Matrix Partners"),
    (r"^nexus( venture partners| ventures)?$", "Nexus Venture Partners"),
    (r"^blume ventures$", "Blume Ventures"),
    (r"^helion venture(s)?( partners?)?$", "Helion Venture Partners"),
    (r"^softbank( group| corp)?$", "SoftBank Group"),
    (r"^lightspeed( venture partners| ventures| india partners| india)( india)?$", "Lightspeed Venture Partners"),
    (r"^indian angel network.*$", "Indian Angel Network")
]

investor_parts = investors_clean.str.split(r",|&| and |;", regex=True).explode().str.strip(" .")
investor_parts = investor_parts[investor_parts.str.len() >= 2]
investor_parts = investor_parts[~investor_parts.str.lower().isin([
    "others", "other investors", "existing investors", "etc", "ltd", "pvt",
    "pvt ltd", "inc", "llc", "llp", "limited", "co"
])]
investor_parts.loc[investor_parts.str.contains("undisclosed", case=False, na=False)] = "Undisclosed Investors"

for alias_pattern, standard_name in investor_aliases:
    investor_parts = investor_parts.str.replace(alias_pattern, standard_name, case=False, regex=True)

investor_count = investor_parts.groupby(level=0).size()
startup["investor_count"] = investor_count.reindex(startup.index).fillna(0).astype(int)

startup[["investors_name", "investor_count"]].head(10)

,investors_name,investor_count
0,Tiger Global Management,1
1,Susquehanna Growth Equity,1
2,Sequoia Capital India,1
3,Vinod Khatumal,1
4,Sprout Venture Partners,1
5,Chiratae Ventures,1
6,Ant Financial,1
7,Sathguru Catalyzer Advisors,1
8,Ping An Global Voyager Fund,1
9,"Mumbai Angels, Ravikanth Reddy",2


### Create a separate investor table

In [17]:
investors_long = startup[[
    "startup_name", "sector", "city", "funding_stage", "funding_date", "amount_usd", "investors_name"
]].copy()

investors_long["investor_name"] = (
    investors_long["investors_name"]
    .str.replace(r"\\(.*?\\)", "", regex=True)
    .str.replace(". ", ", ", regex=False)
    .str.split(r",|&| and |;", regex=True)
)

investors_long = investors_long.explode("investor_name")
investors_long["investor_name"] = investors_long["investor_name"].str.strip(" .")
investors_long = investors_long[investors_long["investor_name"].notna()]
investors_long = investors_long[investors_long["investor_name"].str.len() >= 2]

for alias_pattern, standard_name in investor_aliases:
    investors_long["investor_name"] = investors_long["investor_name"].str.replace(
        alias_pattern, standard_name, case=False, regex=True
    )

investors_long.head()

,startup_name,sector,city,funding_stage,funding_date,amount_usd,investors_name,investor_name
0,BYJU'S,EdTech,Bengaluru,Private Equity,2020-01-09,200000000.0,Tiger Global Management,Tiger Global
1,Shuttl,Logistics & Mobility,Gurugram,Series C,2020-01-13,8048394.0,Susquehanna Growth Equity,Susquehanna Growth Equity
2,MamaEarth,E-commerce & Retail,Bengaluru,Series B,2020-01-09,18358860.0,Sequoia Capital India,Sequoia Capital
3,Www.Wealthbucket.In,FinTech & Financial Services,New Delhi,Series A,2020-01-02,3000000.0,Vinod Khatumal,Vinod Khatumal
4,Fashor,E-commerce & Retail,Mumbai,Seed / Angel,2020-01-02,1800000.0,Sprout Venture Partners,Sprout Venture Partners


### Create date columns

In [18]:
startup["year"] = startup["funding_date"].dt.year
startup["month"] = startup["funding_date"].dt.month
startup["month_name"] = startup["funding_date"].dt.strftime("%b")
startup["quarter"] = "Q" + startup["funding_date"].dt.quarter.astype("Int64").astype(str)
startup["year_month"] = startup["funding_date"].dt.to_period("M").astype(str)

startup[["funding_date", "year", "quarter", "month", "month_name", "year_month"]].head()

,funding_date,year,quarter,month,month_name,year_month
0,2020-01-09,2020.0,Q1,1.0,Jan,2020-01
1,2020-01-13,2020.0,Q1,1.0,Jan,2020-01
2,2020-01-09,2020.0,Q1,1.0,Jan,2020-01
3,2020-01-02,2020.0,Q1,1.0,Jan,2020-01
4,2020-01-02,2020.0,Q1,1.0,Jan,2020-01


### Remove duplicate and invalid-date records

In [19]:
rows_before_cleaning = len(startup)

startup = startup.drop_duplicates(
    subset=["funding_date", "startup_key", "amount_usd", "investors_name", "funding_stage"]
)

startup = startup[startup["funding_date"].notna()].copy()

print("Rows before duplicate/date cleaning:", rows_before_cleaning)
print("Rows after duplicate/date cleaning:", len(startup))
print("Rows removed:", rows_before_cleaning - len(startup))

Rows before duplicate/date cleaning: 3044
Rows after duplicate/date cleaning: 3043
Rows removed: 1


### Create deal-size groups and deal IDs

In [20]:
amount_bins = [0, 1_000_000, 5_000_000, 20_000_000, 100_000_000, np.inf]
amount_labels = ["< $1M", "$1M-5M", "$5M-20M", "$20M-100M", "$100M+"]

startup["deal_size_bucket"] = pd.cut(
    startup["amount_usd"],
    bins=amount_bins,
    labels=amount_labels,
    right=False
).astype("string")

startup.loc[startup["amount_usd"].isna(), "deal_size_bucket"] = "Undisclosed"

startup = startup.sort_values(["funding_date", "sr_no"]).reset_index(drop=True)
startup["deal_id"] = np.arange(1, len(startup) + 1)

startup[["deal_id", "startup_name", "funding_date", "amount_usd", "deal_size_bucket"]].head()

,deal_id,startup_name,funding_date,amount_usd,deal_size_bucket
0,1,Lenskart,2015-01-02,2150000.0,$1M-5M
1,2,VioletStreet,2015-01-02,315000.0,< $1M
2,3,Lookup,2015-01-05,380000.0,< $1M
3,4,#Fame,2015-01-05,10000000.0,$5M-20M
4,5,Gympik,2015-01-05,135000.0,< $1M


### Prepare and save the final cleaned dataset

In [21]:
final_columns = [
    "deal_id", "sr_no", "funding_date", "year", "quarter", "month", "month_name", "year_month",
    "startup_name", "industry_vertical", "sector", "sub_vertical", "city", "is_international_hq",
    "is_multi_city", "investors_name", "investor_count", "investment_type", "funding_stage",
    "amount_usd", "amount_usd_mn", "amount_disclosed", "deal_size_bucket", "remarks"
]

startup_clean = startup[final_columns].copy()
startup_clean["funding_date"] = startup_clean["funding_date"].dt.strftime("%Y-%m-%d")

startup_clean.to_csv("startup_funding_clean.csv", index=False, encoding="utf-8")

print("Cleaned startup funding dataset saved.")
startup_clean.head()

Cleaned startup funding dataset saved.


,deal_id,sr_no,funding_date,year,quarter,month,month_name,year_month,startup_name,industry_vertical,...,is_multi_city,investors_name,investor_count,investment_type,funding_stage,amount_usd,amount_usd_mn,amount_disclosed,deal_size_bucket,remarks
0,1,2990,2015-01-02,2015.0,Q1,1.0,Jan,2015-01,Lenskart,<NA>,...,0,"TPG Growth, TR Capital, IDG Ventures",3,Private Equity,Private Equity,2150000.0,2.15,1,$1M-5M,<NA>
1,2,2991,2015-01-02,2015.0,Q1,1.0,Jan,2015-01,VioletStreet,<NA>,...,0,"Venkat Vallabhneni, Sudhakar Reddy, Srinivasa ...",3,Seed Funding,Seed / Angel,315000.0,0.315,1,< $1M,<NA>
2,3,2992,2015-01-05,2015.0,Q1,1.0,Jan,2015-01,Lookup,<NA>,...,0,"DeNA Co., Japan & Teruhide Sato (in Addition t...",4,Seed Funding,Seed / Angel,380000.0,0.38,1,< $1M,<NA>
3,4,2993,2015-01-05,2015.0,Q1,1.0,Jan,2015-01,#Fame,<NA>,...,0,TO THE NEW Ventures,1,Private Equity,Private Equity,10000000.0,10.0,1,$5M-20M,<NA>
4,5,2994,2015-01-05,2015.0,Q1,1.0,Jan,2015-01,Gympik,<NA>,...,0,"Tom Clausen, Rohan Ajila, Haldyn Glass",3,Seed Funding,Seed / Angel,135000.0,0.135,1,< $1M,<NA>


### Save the investor-level dataset

In [22]:
# Keep the deal ID with each investor record so it can be linked to the main table.
investor_export = startup[[
    "deal_id", "startup_name", "sector", "city", "funding_stage", "year", "amount_usd", "investors_name"
]].copy()

investor_export["investor_name"] = (
    investor_export["investors_name"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.replace(". ", ", ", regex=False)
    .str.split(r",|&| and |;", regex=True)
)

investor_export = investor_export.explode("investor_name")
investor_export["investor_name"] = investor_export["investor_name"].str.strip(" .")
investor_export = investor_export[investor_export["investor_name"].notna()]
investor_export = investor_export[investor_export["investor_name"].str.len() >= 2]

for alias_pattern, standard_name in investor_aliases:
    investor_export["investor_name"] = investor_export["investor_name"].str.replace(
        alias_pattern, standard_name, case=False, regex=True
    )

investor_export = investor_export[[
    "deal_id", "startup_name", "sector", "city", "funding_stage", "year", "amount_usd", "investor_name"
]]

investor_export.to_csv("investors_long.csv", index=False, encoding="utf-8")

print("Investor-level dataset saved.")
investor_export.head()

Investor-level dataset saved.


,deal_id,startup_name,sector,city,funding_stage,year,amount_usd,investor_name
0,1,Lenskart,Unknown,Unknown,Private Equity,2015.0,2150000.0,TPG Growth
0,1,Lenskart,Unknown,Unknown,Private Equity,2015.0,2150000.0,TR Capital
0,1,Lenskart,Unknown,Unknown,Private Equity,2015.0,2150000.0,IDG Ventures
1,2,VioletStreet,Unknown,Unknown,Seed / Angel,2015.0,315000.0,Venkat Vallabhneni
1,2,VioletStreet,Unknown,Unknown,Seed / Angel,2015.0,315000.0,Sudhakar Reddy


### Final checks

In [23]:
print("Cleaned dataset shape:", startup_clean.shape)
print("Investor table shape:", investor_export.shape)

print("\nMissing values in cleaned dataset:")
print(startup_clean.isnull().sum())

print("\nFunding stages:")
print(startup_clean["funding_stage"].value_counts())

print("\nTop sectors by number of funding records:")
print(startup_clean["sector"].value_counts().head(10))

Cleaned dataset shape: (3043, 24)
Investor table shape: (6280, 8)

Missing values in cleaned dataset:
deal_id                   0
sr_no                     0
funding_date              0
year                      0
quarter                   0
month                     0
month_name                0
year_month                0
startup_name              0
industry_vertical       170
sector                    0
sub_vertical            935
city                      0
is_international_hq       0
is_multi_city             0
investors_name            0
investor_count            0
investment_type           4
funding_stage             0
amount_usd              971
amount_usd_mn           971
amount_disclosed          0
deal_size_bucket          0
remarks                2624
dtype: int64

Funding stages:
funding_stage
Seed / Angel                            1541
Private Equity                          1363
Series A                                  33
Debt                                      29
Se